# 在 Middleware 中访问 Static Runtime Context

Middleware 可以根据本次调用的用户、角色、租户或语言动态调整 Agent 行为。

- Node-style hook：通过函数参数 `runtime.context` 访问。
- Wrap-style hook：通过 `request.runtime.context` 访问。
- `dynamic_prompt`：专门用于动态生成 system prompt 的便捷 Middleware。

In [ ]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)


@dataclass(frozen=True)
class AppContext:
    user_id: str
    user_name: str
    user_role: str
    locale: str = "zh-CN"


## Node-style hook

`before_model` 在每次模型调用前执行，适合日志、校验和状态更新。Context 由 `Runtime` 参数提供。

In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime


@before_model
def log_user_context(
    state: AgentState,
    runtime: Runtime[AppContext],
) -> None:
    context = runtime.context
    print(
        f"[before_model] user_id={context.user_id}, "
        f"role={context.user_role}, locale={context.locale}"
    )


## Wrap-style hook

`wrap_model_call` 没有独立的 `runtime` 参数，Runtime 位于 `request.runtime` 中。它可以在模型调用前后执行逻辑，也可以通过 `request.override(...)` 修改本次模型请求。

In [ ]:
from collections.abc import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call


@wrap_model_call
def log_model_request_context(
    request: ModelRequest[AppContext],
    handler: Callable[[ModelRequest[AppContext]], ModelResponse],
) -> ModelResponse:
    context = request.runtime.context
    print(f"[wrap_model_call] 正在为 {context.user_name} 调用模型")
    return handler(request)


## 便捷 Middleware：`dynamic_prompt`

`dynamic_prompt` 是生成动态 system prompt 的专用装饰器。它接收 `ModelRequest`，可以读取 Context，然后直接返回字符串或 `SystemMessage`。

适合根据用户角色、语言、租户配置等生成提示词。该提示词在模型调用时动态构造，不会写入 Agent State 或长期 Store。

In [ ]:
from langchain.agents.middleware import dynamic_prompt


@dynamic_prompt
def context_aware_prompt(request: ModelRequest[AppContext]) -> str:
    context = request.runtime.context

    role_rule = (
        "可以提供管理层面的分析"
        if context.user_role == "admin"
        else "只提供普通用户可访问的信息"
    )

    return (
        "你是一个个人助手。\n"
        f"当前用户姓名：{context.user_name}\n"
        f"用户角色：{context.user_role}，{role_rule}。\n"
        f"请使用语言区域：{context.locale}。"
    )


## 创建并调用 Agent

同一个 Agent 可以在每次调用时接收不同 Context。下面两次调用会生成不同的动态提示词。

In [ ]:
from langchain.agents import create_agent


agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        log_user_context,
        log_model_request_context,
        context_aware_prompt,
    ],
    context_schema=AppContext,
)

contexts = [
    AppContext("user-1", "小花", "member", "zh-CN"),
    AppContext("admin-1", "管理员老王", "admin", "zh-CN"),
]

for context in contexts:
    result = agent.invoke(
        {"messages": "请说明你会如何帮助我。"},
        context=context,
    )
    print(f"\n[{context.user_id}] {result['messages'][-1].content}")


## 小结

- Node-style hooks 使用 `runtime.context`。
- Wrap-style hooks 和 `dynamic_prompt` 使用 `request.runtime.context`。
- `dynamic_prompt` 适合动态 system prompt；更复杂的模型、工具或消息修改使用 `wrap_model_call`。
- Context 是应用传入的可信运行依赖，不应让模型自行构造用户角色或用户 ID。